In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import xml.etree.ElementTree as ET

np.random.seed(42)

# helper to create random dates
def random_dates(n):
    base = datetime(2024,1,1)
    return [base + timedelta(days=random.randint(0,120)) for _ in range(n)]

# helper messy descriptions
descriptions = [
    "UPI Payment", "Amazon Order", "Salary Credit", "Refund",
    "Swiggy", "Zomato", "Netflix", "Electricity Bill",
    "Client Payment", "Razorpay Payment", "ATM Withdrawal",
    "Transfer to savings", "Interest Credit"
]

# ---------------------------------------------------
# 1) EXCEL SALES — messy column names + noise rows
# ---------------------------------------------------
n = 300
df_sales = pd.DataFrame({
    "Order Dt": random_dates(n),
    "Total Paid INR": np.random.randint(100,10000,n),
    "Txn Kind": np.random.choice(["Sale","Refund"],n),
    "Random Notes Column": np.random.choice(descriptions,n),
    "Extra useless column": np.random.randn(n)
})

# add blank rows + duplicates
df_sales = pd.concat([df_sales, df_sales.sample(20)])
df_sales.to_excel("excel_sales.xlsx", index=False)

# ---------------------------------------------------
# 2) PAYMENT GATEWAY CSV — includes failures & fees
# ---------------------------------------------------
n = 350
df_pg = pd.DataFrame({
    "Transaction Date": random_dates(n),
    "Amount INR": np.random.randint(100,8000,n),
    "Status": np.random.choice(["SUCCESS","FAILED","PENDING"],n,p=[0.7,0.2,0.1]),
    "Gateway Fee": np.random.randint(5,80,n),
    "Description": np.random.choice(descriptions,n)
})

df_pg.to_csv("payment_gateway.csv", index=False)

# ---------------------------------------------------
# 3) HDFC STATEMENT — debit credit mess
# ---------------------------------------------------
n = 320
debits = np.random.randint(0,5000,n)
credits = np.random.randint(0,8000,n)

# randomly blank one side
mask = np.random.rand(n) > 0.5
debits[mask] = 0
credits[~mask] = 0

df_hdfc = pd.DataFrame({
    "Txn Date": random_dates(n),
    "Narration": np.random.choice(descriptions,n),
    "Withdrawal Amt": debits,
    "Deposit Amt": credits,
    "Balance (INR)": np.random.randint(1000,90000,n)
})

df_hdfc.to_csv("hdfc_statement.csv", index=False)

# ---------------------------------------------------
# 4) AXIS RAW TXT — inconsistent spacing
# ---------------------------------------------------
with open("axis_statement_raw.txt","w") as f:
    f.write("AXIS BANK ACCOUNT STATEMENT\n\n")
    for i in range(300):
        date = (datetime(2024,1,1) + timedelta(days=random.randint(0,120))).strftime("%d-%m-%y")
        desc = random.choice(descriptions)
        debit = random.choice(["", str(random.randint(100,4000))])
        credit = "" if debit else str(random.randint(100,15000))
        f.write(f"{date}   {desc:<25}   {debit:<8}   {credit}\n")

# ---------------------------------------------------
# 5) TALLY XML — missing narration randomly
# ---------------------------------------------------
root = ET.Element("ENVELOPE")
body = ET.SubElement(root,"BODY")
data = ET.SubElement(body,"DATA")

for i in range(250):
    voucher = ET.SubElement(data,"VOUCHER")
    ET.SubElement(voucher,"DATE").text = (datetime(2024,1,1)+timedelta(days=random.randint(0,120))).strftime("%Y%m%d")
    ET.SubElement(voucher,"AMOUNT").text = str(random.randint(-5000,15000))
    ET.SubElement(voucher,"VOUCHERTYPENAME").text = random.choice(["Sales","Purchase","Receipt"])

    # narration missing randomly
    if random.random() > 0.4:
        ET.SubElement(voucher,"NARRATION").text = random.choice(descriptions)

tree = ET.ElementTree(root)
tree.write("tally_export.xml")

print("All messy test files generated.")


All messy test files generated.


In [10]:
import pandas as pd
# Load the Excel sales data
df_sales = pd.read_excel("excel_sales.xlsx")
# Load the payment gateway CSV
df_pg = pd.read_csv("payment_gateway.csv")
# Load the HDFC statement CSV
df_hdfc = pd.read_csv("hdfc_statement.csv")
# Load the Axis raw text file
with open("axis_statement_raw.txt", "r") as f:
    axis_data = f.readlines()[2:]  # Skip header lines
df_tally = pd.read_xml("tally_export.xml", xpath=".//VOUCHER")


In [ ]:
df_tally


,DATE,AMOUNT,VOUCHERTYPENAME,NARRATION
0,20240423,-1617,Purchase,Salary Credit
1,20240131,165,Sales,Client Payment
2,20240220,8524,Receipt,None
3,20240424,-458,Purchase,Amazon Order
4,20240102,12078,Receipt,None
...,...,...,...,...
245,20240228,-1250,Receipt,Razorpay Payment
246,20240303,11549,Receipt,None
247,20240223,8789,Purchase,Razorpay Payment
248,20240207,9767,Receipt,Transfer to savings


In [4]:
df_sales


,Order Dt,Total Paid INR,Txn Kind,Random Notes Column,Extra useless column
0,2024-02-23,7370,Sale,Amazon Order,0.968628
1,2024-01-11,960,Refund,UPI Payment,0.645733
2,2024-02-23,5490,Sale,Netflix,1.212156
3,2024-04-30,5291,Refund,Electricity Bill,-0.144049
4,2024-01-18,5834,Sale,Netflix,0.128041
...,...,...,...,...,...
315,2024-03-29,1595,Sale,Swiggy,2.976197
316,2024-04-12,5955,Sale,Zomato,-0.439864
317,2024-03-25,8780,Refund,ATM Withdrawal,0.429448
318,2024-03-11,325,Sale,Amazon Order,-0.121447


In [ ]:
df_pg


,Transaction Date,Amount INR,Status,Gateway Fee,Description
0,2024-01-28,2112,FAILED,72,Transfer to savings
1,2024-03-23,1647,SUCCESS,49,Electricity Bill
2,2024-03-13,394,SUCCESS,64,Razorpay Payment
3,2024-04-01,2085,SUCCESS,16,Zomato
4,2024-02-27,5443,SUCCESS,36,Razorpay Payment
...,...,...,...,...,...
345,2024-04-18,503,FAILED,28,Interest Credit
346,2024-04-10,7048,SUCCESS,34,Netflix
347,2024-04-05,4711,SUCCESS,11,Client Payment
348,2024-01-30,2879,SUCCESS,39,ATM Withdrawal


In [ ]:
df_hdfc


,Txn Date,Narration,Withdrawal Amt,Deposit Amt,Balance (INR)
0,2024-03-04,Netflix,0,6207,19257
1,2024-03-19,Razorpay Payment,1531,0,13855
2,2024-04-08,Netflix,4882,0,86178
3,2024-03-08,Interest Credit,2990,0,21025
4,2024-03-18,Client Payment,0,266,5458
...,...,...,...,...,...
315,2024-04-19,Transfer to savings,2660,0,21324
316,2024-02-01,UPI Payment,2211,0,23867
317,2024-03-01,Zomato,1909,0,1203
318,2024-01-07,Swiggy,3730,0,17544


In [9]:
axis_data


['12-01-24   ATM Withdrawal              2727       \n',
 '19-01-24   Salary Credit               449        \n',
 '02-03-24   Razorpay Payment                       2400\n',
 '18-01-24   Razorpay Payment            2364       \n',
 '09-04-24   ATM Withdrawal                         8885\n',
 '06-02-24   ATM Withdrawal                         1025\n',
 '19-02-24   Electricity Bill                       6132\n',
 '29-04-24   Transfer to savings                    2069\n',
 '31-01-24   Salary Credit               2238       \n',
 '04-03-24   Transfer to savings         1009       \n',
 '28-04-24   Netflix                                5456\n',
 '17-04-24   ATM Withdrawal                         14382\n',
 '06-01-24   ATM Withdrawal                         6743\n',
 '02-03-24   Electricity Bill                       6106\n',
 '31-01-24   Razorpay Payment            3475       \n',
 '18-01-24   UPI Payment                 3055       \n',
 '02-01-24   Client Payment                        

In [16]:
import pandas as pd

# Check 1 — Payment gateway: how many FAILED?
df_pg = pd.read_csv('payment_gateway.csv')
print("Failed transactions:", len(df_pg[df_pg['Status'] == 'FAILED']))

# Check 2 — HDFC: confirm no single Amount column
df_hdfc = pd.read_csv('hdfc_statement.csv')
print("HDFC columns:", df_hdfc.columns.tolist())
print("'Amount' in columns:", 'Amount' in df_hdfc.columns)



# ---

# ## One Question

# Look at your Axis data:
# ```
'12-01-24   ATM Withdrawal    2727       \n'
'02-03-24   Razorpay Payment             2400\n'


Failed transactions: 59
HDFC columns: ['Txn Date', 'Narration', 'Withdrawal Amt', 'Deposit Amt', 'Balance (INR)']
'Amount' in columns: False


'02-03-24   Razorpay Payment             2400\n'

In [22]:
import xml.etree.ElementTree as ET

tree = ET.parse('tally_export.xml')
root = tree.getroot()

narration_missing = 0
total_vouchers = 0

for voucher in root.iter('VOUCHER'):
    total_vouchers += 1
    narration = voucher.find('NARRATION')
    if narration is None or narration.text is None:
        narration_missing += 1

print(f"Total vouchers: {total_vouchers}")
print(f"Missing NARRATION: {narration_missing}")
# ```

# Run it. Paste the output.

# ---

# ## Also Answer The Axis Question

# You still haven't answered this:
# ```
# '12-01-24   ATM Withdrawal    2727          \n'  ← debit
# '02-03-24   Razorpay Payment               2400\n'  ← credit


Total vouchers: 250
Missing NARRATION: 105


In [1]:
print("FILE BEING READ:", config_path.resolve())


NameError: name 'config_path' is not defined